# Week 1 Forecast Workflow

This notebook wraps the current week-1 forecasting pipeline implemented in `forecast_week1.py`.

Use it to:

- configure API and token
- pull `test_world` data
- run backtests and final forecast
- save output files
- inspect weights, bucket weights, and backtest results


## 1. Environment

Install dependencies only if you have not installed them yet.

In [ ]:
# Optional: install dependencies if needed
# %pip install pandas numpy requests scikit-learn lightgbm statsmodels

## 2. Import project code

The notebook reuses the functions already implemented in `forecast_week1.py` instead of copying model logic into notebook cells.

In [ ]:
from pathlib import Path
import json
import os
import sys

import pandas as pd

PROJECT_DIR = Path.cwd()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from forecast_week1 import (
    DEFAULT_API_SERVER,
    DEFAULT_HOLDOUT_ORIGINS,
    DEFAULT_HORIZON,
    DEFAULT_SCENARIO,
    DEFAULT_BACKTEST_ORIGINS,
    generate_forecast_from_api,
    generate_forecast_from_csv,
    save_outputs,
)

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 160)

PROJECT_DIR

## 3. API configuration

Recommended: set your token in the `RETAIL_ANALYTICS_ACCESS_TOKEN` environment variable.

You can also paste it directly into `ACCESS_TOKEN` for a temporary notebook run.

In [ ]:
API_SERVER = DEFAULT_API_SERVER
SCENARIO_NAME = DEFAULT_SCENARIO
OUTPUT_DIR = PROJECT_DIR / 'outputs'

ACCESS_TOKEN = os.getenv('RETAIL_ANALYTICS_ACCESS_TOKEN', '')

# Fallback: paste token directly here if needed
# ACCESS_TOKEN = 'your_token_here'

print('API_SERVER =', API_SERVER)
print('SCENARIO_NAME =', SCENARIO_NAME)
print('OUTPUT_DIR =', OUTPUT_DIR)
print('Token loaded =', bool(ACCESS_TOKEN))

## 4. Run API forecast

This cell will:

- pull course API data
- run multi-model backtests
- search overall weights and bucket weights
- generate the final week-1 forecast
- save submission and analysis files


In [ ]:
artifacts = generate_forecast_from_api(
    access_token=ACCESS_TOKEN,
    api_server=API_SERVER,
    scenario_name=SCENARIO_NAME,
    horizon=DEFAULT_HORIZON,
    max_origins=DEFAULT_BACKTEST_ORIGINS,
    holdout_origins=DEFAULT_HOLDOUT_ORIGINS,
)

feature_date = artifacts.history['date'].max().strftime('%Y-%m-%d')
paths = save_outputs(artifacts, OUTPUT_DIR, feature_date)

feature_date, paths

## 5. Core outputs

Inspect the selected weights, bucket weights, and a quick view of forecast totals.

In [ ]:
print('Selected weights:')
print(json.dumps(artifacts.selected_weights, indent=2, ensure_ascii=False))

print('\nSelected bucket weights:')
print(json.dumps(artifacts.selected_bucket_weights, indent=2, ensure_ascii=False))

print('\nForecast panel preview:')
display(artifacts.forecast_panel.head())

print('\n28-day total forecast by option (top 10):')
display(
    artifacts.forecast_panel.groupby('option_id', as_index=False)['forecast']
    .sum()
    .sort_values('forecast', ascending=False)
    .head(10)
)

## 6. Overall backtest summary

Focus on `ALL_SELECTION` and `ALL_HOLDOUT` first.

In [ ]:
backtest_summary = artifacts.backtest_summary.copy()
display(backtest_summary)

print('ALL_SELECTION / ALL_HOLDOUT:')
display(
    backtest_summary[backtest_summary['origin'].isin(['ALL_SELECTION', 'ALL_HOLDOUT'])]
    .sort_values(['split', 'score'], ascending=[True, False])
)

## 7. Horizon-level analysis

Use this to inspect which model wins at different horizons.

In [ ]:
horizon_metrics = artifacts.backtest_metrics_by_horizon.copy()

holdout_h = horizon_metrics[horizon_metrics['origin'] == 'ALL_HOLDOUT'].copy()
display(holdout_h.head(20))

best_by_h = holdout_h.loc[holdout_h.groupby('horizon')['score'].idxmax()]
display(best_by_h[['horizon', 'model', 'score']].sort_values('horizon'))

## 8. Bucket-level analysis

Compare bucket performance with the selected bucket weights.

In [ ]:
bucket_metrics = artifacts.backtest_metrics_by_bucket.copy()

display(
    bucket_metrics[bucket_metrics['origin'] == 'ALL_HOLDOUT']
    .sort_values(['bucket', 'score'], ascending=[True, False])
)

## 9. Reload saved files

If you reopen the notebook later, you can load saved outputs directly from `outputs` without retraining.

In [ ]:
saved_backtest = pd.read_csv(paths['backtest'])
saved_forecast = pd.read_csv(paths['forecast'])

display(saved_backtest.tail(12))
display(saved_forecast.head())

## 10. Offline CSV mode example

Use this if you want to run the pipeline from a local history file instead of the API.

In [ ]:
# sample_history_csv = PROJECT_DIR / 'sample_history.csv'
# offline_artifacts = generate_forecast_from_csv(sample_history_csv, horizon=DEFAULT_HORIZON)
# offline_feature_date = offline_artifacts.history['date'].max().strftime('%Y-%m-%d')
# offline_paths = save_outputs(offline_artifacts, OUTPUT_DIR, offline_feature_date)
# offline_paths

## 11. Equivalent CLI command

```powershell
python forecast_week1.py --mode api --access-token "<YOUR_TOKEN>" --output-dir .\outputs
```